In [4]:
!pip install chromadb sentence-transformers transformers gradio pandas

In [14]:
import os
import json
import pandas as pd
from typing import List, Dict, Any

# This is for Embedding model
from sentence_transformers import SentenceTransformer

# This is Vector DB
import chromadb
from chromadb.config import Settings

# Generation model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# User Interface
import gradio as gr


DEFAULT_CSV_PATH = "myproject_dataset_1000 (2).csv"
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
GEN_MODEL_NAME = 'google/flan-t5-small'
TOP_K = 3



def load_csv(path: str) -> pd.DataFrame:

    df = pd.read_csv(path)

    df.columns = [c.strip().lower().replace(' ', '_').replace('\n','_') for c in df.columns]

    mapping = {}
    for c in df.columns:
        if 'id' in c:
            mapping[c] = 'medicine_id'
        elif 'name' in c:
            mapping[c] = 'medicine_name'
        elif 'strength' in c:
            mapping[c] = 'strength'
        elif 'use' in c and 'case' in c:
            mapping[c] = 'use_case'
        elif 'alternative' in c:
            mapping[c] = 'alternative'
        elif 'stock' in c:
            mapping[c] = 'stock'
        elif 'dosage' in c:
            mapping[c] = 'dosage_instructions'

    df = df.rename(columns=mapping)

    # these are the columns are there in the dataset
    for col in ['medicine_id', 'medicine_name', 'strength', 'use_case', 'alternative', 'stock', 'dosage_instructions']:
        if col not in df.columns:
            df[col] = ''

    # Cleaning the data
    df['medicine_name'] = df['medicine_name'].astype(str).str.strip()
    df['stock'] = df['stock'].astype(str).str.strip().str.lower().replace({'yes':'yes','no':'no'})

    return df


def build_documents_from_df(df: pd.DataFrame) -> List[Dict[str, Any]]:
    """Create text documents from each row for indexing and retrieval."""
    docs = []
    for i, row in df.iterrows():
        text = f"Medicine: {row.get('medicine_name','')} | Strength: {row.get('strength','')} | Use case: {row.get('use_case','')} | Alternatives: {row.get('alternative','')} | Stock: {row.get('stock','')} | Dosage: {row.get('dosage_instructions','')}"
        metadata = {
            'medicine_name': row.get('medicine_name',''),
            'strength': row.get('strength',''),
            'use_case': row.get('use_case',''),
            'alternative': row.get('alternative',''),
            'stock': row.get('stock',''),
            'dosage_instructions': row.get('dosage_instructions','')
        }
        docs.append({'id': str(i), 'text': text, 'metadata': metadata})
    return docs

# Buildiing embedding model and chroma client

def init_embedding_model(model_name: str = EMBEDDING_MODEL_NAME):
    print(f"Loading embedding model: {model_name} ...")
    emb = SentenceTransformer(model_name)
    return emb


def init_gen_model(model_name: str = GEN_MODEL_NAME):
    print(f"Loading generation model: {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    return tokenizer, model


def init_chroma(client_settings: Dict = None):
    settings = Settings(anonymized_telemetry=False)
    client = chromadb.Client(settings)
    return client


def create_or_rebuild_index(df: pd.DataFrame, embedding_model) -> Any:
    """Creates a Chroma collection, computes embeddings for documents, and stores them."""
    docs = build_documents_from_df(df)
    client = init_chroma()


    col_name = 'medicines'
    try:
        collection = client.get_collection(name=col_name)

        client.delete_collection(name=col_name)
    except Exception:
        pass
    collection = client.create_collection(name=col_name)

    # create embeddings for each doc
    texts = [d['text'] for d in docs]
    print('Creating embeddings for', len(texts), 'documents...')
    embeddings = embedding_model.encode(texts, show_progress_bar=True)

    ids = [d['id'] for d in docs]
    metadatas = [d['metadata'] for d in docs]

    collection.add(ids=ids, documents=texts, metadatas=metadatas, embeddings=embeddings.tolist())
    print('Index built.')
    return client, collection

# Retrieval + Generation pipeline

def retrieve_top_k(collection, embedding_model, query: str, top_k: int = TOP_K):
    q_emb = embedding_model.encode([query])[0].tolist()
    results = collection.query(query_embeddings=[q_emb], n_results=top_k, include=['metadatas','documents','distances'])
    hits = []
    if 'ids' in results and len(results['ids'])>0:
        for idx in range(len(results['ids'][0])):
            hit = {
                'id': results['ids'][0][idx],
                'document': results['documents'][0][idx],
                'metadata': results['metadatas'][0][idx],
                'distance': results['distances'][0][idx]
            }
            hits.append(hit)
    return hits


def build_prompt(query: str, hits: List[Dict], user_role: str = 'pharmacist') -> str:
    """Construct a prompt for the generator including retrieved facts and instructions.
    Keep the prompt concise to fit small models.
    """
    retrieved_texts = '\n'.join([f"- {h['document']}" for h in hits])
    prompt = (
        f"You are a helpful {user_role} assistant for a hospital/medical shop. "
        f"Use only the facts provided below from the shop's dataset to answer the user's query. "
        f"If asked about stock, indicate availability exactly as 'in stock' or 'out of stock'. "
        f"If an alternative is listed and the main item is out of stock, suggest it. "
        f"Always remind to consult a doctor for dosing if it is a prescription or severe condition. "
        f"Keep the answer concise (max 120 words).\n\n"
        f"Facts:\n{retrieved_texts}\n\n"
        f"User question: {query}\n\n"
        f"Answer:"
    )
    return prompt



def generate_answer(tokenizer, model, prompt: str, max_length: int = 200) -> str:
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, padding='longest')
    out = model.generate(**inputs, max_new_tokens=max_length, do_sample=False)
    answer = tokenizer.decode(out[0], skip_special_tokens=True)
    return answer


def run_agent(collection, embedding_model, tokenizer, model, df: pd.DataFrame, query: str) -> str:
    hits = retrieve_top_k(collection, embedding_model, query, top_k=TOP_K)

    q_lower = query.lower()
    detected_row = None
    for i, row in df.iterrows():
        name = str(row['medicine_name']).lower()
        if name and name in q_lower:
            detected_row = row
            break

    preface_lines = []
    if detected_row is not None and detected_row['medicine_name'] != '':
        stock = str(detected_row.get('stock','')).strip().lower()
        if stock.startswith('y'):
            preface_lines.append(f"Stock status: in stock")
        elif stock.startswith('n'):
            preface_lines.append(f"Stock status: out of stock")
            alt = str(detected_row.get('alternative','')).strip()
            if alt:
                preface_lines.append(f"Alternative: {alt}")

    prompt = build_prompt(query, hits)
    if preface_lines:
        prompt = 'Additional shop facts: ' + '; '.join(preface_lines) + '\n\n' + prompt

    answer = generate_answer(tokenizer, model, prompt)

    if detected_row is not None:
        stock = str(detected_row.get('stock','')).strip().lower()
        stock_text = 'in stock' if stock.startswith('y') else 'out of stock'
        alt = str(detected_row.get('alternative','')).strip()
        dosage = str(detected_row.get('dosage_instructions','')).strip()
        response = f"Yes, {detected_row['medicine_name']} {detected_row.get('strength','')} is used for {detected_row.get('use_case','').lower()}."

        if stock_text == 'in stock':
          response += " It is currently in stock."
        else:
          response += " However, it is currently out of stock."
          if alt:
            response += f" An alternative available is {alt}."

        if dosage:
          response += f" Usual dosage is {dosage}, but please consult a doctor before use."

        return response


def main(csv_path: str = DEFAULT_CSV_PATH):

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found at {csv_path}. Upload your CSV or change path.")
    df = load_csv(csv_path)


    embed_model = init_embedding_model()
    tokenizer, gen_model = init_gen_model()


    client, collection = create_or_rebuild_index(df, embed_model)


    def answer_fn(user_query, uploaded_file=None):
        nonlocal client, collection, df

        if uploaded_file is not None:

            new_df = load_csv(uploaded_file.name)
            df = new_df
            client, collection = create_or_rebuild_index(df, embed_model)


        try:
            resp = run_agent(collection, embed_model, tokenizer, gen_model, df, user_query)
        except Exception as e:
            resp = f"Error while answering: {e}"
        return resp

    with gr.Blocks() as demo:
        gr.Markdown("# AI Agent — Hospital & Medical Shop (RAG)")
        gr.Markdown("Upload a CSV (optional) or use the provided dataset. Ask queries like 'Do you have Paracetamol 500mg?'")
        with gr.Row():
            inp = gr.Textbox(placeholder='Type your question here', label='User Query')
            file_in = gr.File(label='Upload CSV (optional)')
        out = gr.Textbox(label='Agent Response', lines=8)
        submit = gr.Button('Ask')
        submit.click(answer_fn, inputs=[inp, file_in], outputs=[out])

    demo.launch(share=True)


if __name__ == '__main__':

    print('The code is working')

The code is working
Creating embeddings for 1009 documents...


Batches: 100%|██████████| 32/32 [00:09<00:00,  3.32it/s]


Index built.
Creating embeddings for 1009 documents...


Batches: 100%|██████████| 32/32 [00:10<00:00,  2.92it/s]


Index built.
Creating embeddings for 1009 documents...


...

Index built.
Creating embeddings for 1009 documents...


Batches: 100%|██████████| 32/32 [00:13<00:00,  2.43it/s]


Index built.
Creating embeddings for 1009 documents...


Batches: 100%|██████████| 32/32 [00:13<00:00,  2.42it/s]


Index built.
Creating embeddings for 1009 documents...


In [13]:
main()


Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5297.27it/s]


Loading generation model: google/flan-t5-small ...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 7918.26it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Creating embeddings for 1009 documents...


Batches: 100%|██████████| 32/32 [00:07<00:00,  4.17it/s]


Index built.
* Running on local URL:  http://127.0.0.1:7862

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
